# Segmentation alimentaire avec FoodSeg103 et YOLOv11-seg

## Objectif

Ce notebook entraîne un modèle de segmentation alimentaire à partir du dataset FoodSeg103.

L’objectif est de segmenter les zones alimentaires dans une image afin d’estimer la proportion occupée par les aliments dans un plat.

## Dataset

Source : FoodSeg103 sur Kaggle  
Chemin Kaggle : `/kaggle/input/datasets/fontainenathan/foodseg103`

Le dataset contient :
- des images dans `Images/img_dir`
- des masques de segmentation dans `Images/ann_dir`
- les listes d’entraînement et de test dans `ImageSets`

## Sorties

Le notebook génère :
- un dataset converti au format YOLO segmentation
- un modèle entraîné `best.pt`
- des résultats de validation

In [ ]:
# Taille du sous-échantillon
# Mettre None pour utiliser le dataset complet
LIMIT_TRAIN = None

In [ ]:
# Désinstaller les versions incompatibles déjà présentes dans le runtime Kaggle.
# Cela évite les conflits avec la version CUDA actuelle.
!pip uninstall -y torch torchvision torchaudio ultralytics

# Installer une version de PyTorch compatible avec le GPU Tesla P100 (architecture sm_60).
# On utilise ici CUDA 12.6, qui prend encore en charge cette génération de GPU.
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126

# Installer Ultralytics sans réinstaller ou modifier PyTorch.

# Le flag --no-deps est important pour conserver notre version compatible.
!pip install ultralytics --no-deps

# Vérification de l’environnement CUDA et du GPU utilisé.
import torch

print("Version PyTorch :", torch.__version__)
print("Version CUDA :", torch.version.cuda)
print("GPU détecté :", torch.cuda.get_device_name(0))
print("Capability CUDA :", torch.cuda.get_device_capability(0))

In [ ]:
# Import des librairies nécessaires.
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from IPython.display import FileLink

In [ ]:
# Chemin racine du dataset FoodSeg103.
ROOT = "/kaggle/input/datasets/cleristonmartinelo/foodsegmentation"

# Dossiers des images et des masques.
IMG_DIR = f"{ROOT}/Images/img_dir"
ANN_DIR = f"{ROOT}/Images/ann_dir"

# Fichier des catégories.
CATEGORY_FILE = f"{ROOT}/category_id.txt"

# Listes train/test.
TRAIN_LIST = f"{ROOT}/ImageSets/train.txt"
TEST_LIST = f"{ROOT}/ImageSets/test.txt"

# Dossier de sortie converti au format YOLO.
DEST = "/kaggle/working/dataset"

print("ROOT:", ROOT)
print("IMG_DIR:", IMG_DIR)
print("ANN_DIR:", ANN_DIR)
print("CATEGORY_FILE:", CATEGORY_FILE)
print("TRAIN_LIST:", TRAIN_LIST)
print("TEST_LIST:", TEST_LIST)


In [ ]:
import time
import random

# Lecture des fichiers train.txt et test.txt.
def lire_liste(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_files = lire_liste(TRAIN_LIST)
val_files = lire_liste(TEST_LIST)

# Sauvegarde des tailles originales
original_train = len(train_files)
original_val = len(val_files)

# Génère automatiquement une seed différente à chaque exécution
SEED = int(time.time()) % 100000
print("Seed utilisée : ", SEED)

random.seed(SEED)

# Mélange aléatoire des données
random.shuffle(train_files)
random.shuffle(val_files)

if LIMIT_TRAIN is not None:
    # Calcul du ratio validation/train original
    val_ratio = original_val / original_train

    # Nombre proportionnel pour validation
    LIMIT_VAL = int(LIMIT_TRAIN * val_ratio)

    # Réduction du dataset
    train_files = train_files[:LIMIT_TRAIN]
    val_files = val_files[:LIMIT_VAL]

    print("Mode réduit activé")
else:
    print("Mode dataset complet activé")

print("Train original :", original_train)
print("Val original :", original_val)
print("Train utilisé :", len(train_files))
print("Val utilisé :", len(val_files))
print("Ratio final :", round(len(val_files) / len(train_files), 4))

print("Exemples train :", train_files[:5])
print("Exemples val :", val_files[:5])

In [ ]:
# Lecture des catégories FoodSeg103.
# Le masque utilise : 0 = background, 1 à 103 = classes alimentaires.
# YOLO doit utiliser : 0 à 102 = classes alimentaires, sans background.

categories_foodseg = {}

with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)  # sépare l'ID et le nom, même si le nom contient des espaces
        if len(parts) == 2:
            id_foodseg = int(parts[0])
            nom_classe = parts[1].strip()
            categories_foodseg[id_foodseg] = nom_classe

# On ignore le background et on décale les IDs pour YOLO.
categories_yolo = {
    id_foodseg - 1: nom
    for id_foodseg, nom in categories_foodseg.items()
    if id_foodseg != 0
}

print("Nombre de classes YOLO:", len(categories_yolo))
print("Exemples:")
for i in list(categories_yolo.keys())[:10]:
    print(i, "->", categories_yolo[i])

# Exemple important : FoodSeg 66 = rice, donc YOLO 65 = rice.
print("FoodSeg 48 -> YOLO", 48 - 1, "=", categories_foodseg.get(48))
print("FoodSeg 66 -> YOLO", 66 - 1, "=", categories_foodseg.get(66))
print("FoodSeg 90 -> YOLO", 90 - 1, "=", categories_foodseg.get(90))


In [ ]:
# Création de la structure attendue par YOLO.
for d in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{DEST}/{d}", exist_ok=True)

print("Structure YOLO créée.")

In [ ]:
# Conversion d'un masque PNG en fichier label YOLO segmentation multi-classes.
# Important : on conserve les vraies catégories du masque FoodSeg103.
# 0 = background, ignoré.
# 1 à 103 = classes alimentaires FoodSeg103.
# YOLO attend des classes de 0 à 102, donc : classe_yolo = valeur_masque - 1.

def mask_to_yolo_multiclass(mask_path, output_txt, min_area=20):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return False

    h, w = mask.shape
    lignes = []

    valeurs_classes = np.unique(mask)

    for valeur in valeurs_classes:
        if valeur == 0:
            continue  # background

        classe_yolo = int(valeur) - 1

        # Sécurité : ignorer une classe inconnue.
        if classe_yolo not in categories_yolo:
            continue

        masque_classe = (mask == valeur).astype(np.uint8) * 255

        contours, _ = cv2.findContours(
            masque_classe,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < 3:
                continue

            if cv2.contourArea(contour) < min_area:
                continue

            # Simplification légère pour éviter des polygones trop longs.
            epsilon = 0.001 * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, epsilon, True)

            if len(contour) < 3:
                continue

            points = contour.reshape(-1, 2)
            coords = []

            for x, y in points:
                coords.append(x / w)
                coords.append(y / h)

            ligne = str(classe_yolo) + " " + " ".join([f"{c:.6f}" for c in coords])
            lignes.append(ligne)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lignes))

    return len(lignes) > 0


In [ ]:
# Fonction pour trouver le masque correspondant à une image.
# Exemple : 00000001.jpg -> 00000001.png

def trouver_masque(nom_image, split):
    base = Path(nom_image).stem
    mask_path = f"{ANN_DIR}/{split}/{base}.png"

    if os.path.exists(mask_path):
        return mask_path

    return None


# Conversion d'un split train ou test vers la structure YOLO.
def convertir_split(files, src_split, dst_split):
    ok = 0
    manquants = 0
    sans_masque_valide = 0

    for nom_image in files:
        img_path = f"{IMG_DIR}/{src_split}/{nom_image}"
        mask_path = trouver_masque(nom_image, src_split)

        if not os.path.exists(img_path) or mask_path is None:
            manquants += 1
            continue

        base = Path(nom_image).stem

        dst_img = f"{DEST}/images/{dst_split}/{nom_image}"
        dst_label = f"{DEST}/labels/{dst_split}/{base}.txt"

        shutil.copy(img_path, dst_img)

        if mask_to_yolo_multiclass(mask_path, dst_label):
            ok += 1
        else:
            sans_masque_valide += 1

    print(f"{dst_split} ok:", ok)
    print(f"{dst_split} manquants:", manquants)
    print(f"{dst_split} sans masque valide:", sans_masque_valide)


convertir_split(train_files, "train", "train")
convertir_split(val_files, "test", "val")


In [ ]:
# Vérification finale de la conversion.
print("Images train:", len(os.listdir(f"{DEST}/images/train")))
print("Labels train:", len(os.listdir(f"{DEST}/labels/train")))
print("Images val:", len(os.listdir(f"{DEST}/images/val")))
print("Labels val:", len(os.listdir(f"{DEST}/labels/val")))

print("Exemples labels:", os.listdir(f"{DEST}/labels/train")[:5])

In [ ]:
# Vérification du contenu d'un fichier label.
exemple_label = os.listdir(f"{DEST}/labels/train")[0]

with open(f"{DEST}/labels/train/{exemple_label}", "r") as f:
    print(f.read()[:500])

In [ ]:
# Création du fichier de configuration YOLO multi-classes.
# On utilise les 103 classes alimentaires de FoodSeg103, sans le background.

yaml_content = f"""
path: {DEST}
train: images/train
val: images/val

names:
"""

for idx in sorted(categories_yolo.keys()):
    nom = categories_yolo[idx].replace("'", "")
    yaml_content += f"  {idx}: '{nom}'\n"

with open("/kaggle/working/dataset.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(yaml_content[:1500])


In [ ]:
# Chargement du modèle YOLO segmentation pré-entraîné.
# Pour tester plus vite, tu peux garder yolo11n-seg.pt.
# Pour un meilleur résultat final, tu peux utiliser yolo11m-seg.pt.

model = YOLO("yolo11n-seg.pt")

# Entraînement du modèle multi-classes.
#model.train(
#    data="/kaggle/working/dataset.yaml",
#    task="segment",
#    epochs=50,
#    imgsz=640,
#    batch=8
#)


# Entraînement rapide pour validation technique
model.train(
    data="/kaggle/working/dataset.yaml",
    task="segment",
    epochs=1,       # au lieu de 50
    imgsz=320,      # au lieu de 640
    batch=16,       # plus grand batch si la VRAM permet
    workers=2,      # réduit le coût de chargement
    cache=True      # garde les images en mémoire
)

In [ ]:
metrics = model.val()

new_map = metrics.box.map50 if hasattr(metrics, "box") else metrics.seg.map50

print("New model mAP50:", new_map)



In [ ]:
import json
import os

old_map = 0

if os.path.exists("/kaggle/input/current-model/metrics.json"):
    with open("/kaggle/input/current-model/metrics.json", "r") as f:
        old_map = json.load(f)["mAP50"]

print("Current production mAP50:", old_map)

In [ ]:
# Test sur des images de validation.
model.predict(
    source=f"{DEST}/images/val",
    conf=0.25,
    save=True,
    max_det=10
)

In [ ]:
# Trouver le modèle entraîné.
!find /kaggle/working -name "best.pt"

In [ ]:
import shutil
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# Lecture des résultats détaillés du training
results = pd.read_csv("/kaggle/working/runs/segment/train/results.csv")

# Colonnes correctes pour segmentation YOLO
best_row = results.loc[results["metrics/seg(mAP50)"].idxmax()]

run_metrics = pd.DataFrame([{
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "sample_size": LIMIT_TRAIN if LIMIT_TRAIN is not None else original_train,
    "val_size": len(val_files),
    "epochs": len(results),
    "best_box_map50": float(best_row["metrics/mAP50(B)"]),
    "best_box_map50_95": float(best_row["metrics/mAP50-95(B)"]),
    "best_mask_map50": float(best_row["metrics/seg(mAP50)"]),
    "best_mask_map50_95": float(best_row["metrics/seg(mAP50-95)"]),
    "promoted": new_map > old_map
}])

if new_map > old_map:
    print("New model is better. Promoting.")

    # Promotion du modèle
    !cp /kaggle/working/runs/segment/train*/weights/best.pt /kaggle/working/best.pt

    # Sauvegarde de la métrique actuelle
    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump({"mAP50": float(new_map)}, f)

else:
    print("New model is worse. Keeping current production model.")

# Fichier global cumulatif
history_path = Path("/kaggle/working/all_runs_metrics.csv")

# Ajouter au fichier existant ou créer un nouveau
if history_path.exists():
    history = pd.read_csv(history_path)
    history = pd.concat([history, run_metrics], ignore_index=True)
else:
    history = run_metrics

# Sauvegarde du fichier cumulatif
history.to_csv(history_path, index=False)

print("Historique global sauvegardé.")
print(history.tail())

# Si pire modèle, on arrête après avoir logué
if new_map <= old_map:
    raise SystemExit()

In [ ]:
# Sauvegarde du fichier cumulatif dans les outputs Kaggle
!mkdir -p /kaggle/working/output
!cp /kaggle/working/all_runs_metrics.csv /kaggle/working/output/